<a href="https://colab.research.google.com/github/VeenusDadri/training/blob/master/advance_pyspark/UDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Spark SQL').getOrCreate()
sc = spark.sparkContext

In [35]:
data=[(1,"maheer", 2000,500),(2,"mahi",4000,1000)]
schema=['id','name', 'salary','bonus']
df=spark.createDataFrame(data,schema)
df.show()

+---+------+------+-----+
| id|  name|salary|bonus|
+---+------+------+-----+
|  1|maheer|  2000|  500|
|  2|  mahi|  4000| 1000|
+---+------+------+-----+



In [36]:
def total_pay(s,b):
  return s+b
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
Total_Payment= udf(lambda x,y : total_pay(x,y), IntegerType())


In [37]:
df.withColumn('Total_Pay',Total_Payment(df.salary,df.bonus)).show()

+---+------+------+-----+---------+
| id|  name|salary|bonus|Total_Pay|
+---+------+------+-----+---------+
|  1|maheer|  2000|  500|     2500|
|  2|  mahi|  4000| 1000|     5000|
+---+------+------+-----+---------+



In [38]:

@udf(returnType=IntegerType())
def total_pay(s,b):
  return s+b



In [39]:
df.select('*', total_pay(df.salary,df.bonus)).show()

+---+------+------+-----+------------------------+
| id|  name|salary|bonus|total_pay(salary, bonus)|
+---+------+------+-----+------------------------+
|  1|maheer|  2000|  500|                    2500|
|  2|  mahi|  4000| 1000|                    5000|
+---+------+------+-----+------------------------+



In [40]:
df.select('*', total_pay(df.salary,df.bonus).alias("Tot_Pay")).show()

+---+------+------+-----+-------+
| id|  name|salary|bonus|Tot_Pay|
+---+------+------+-----+-------+
|  1|maheer|  2000|  500|   2500|
|  2|  mahi|  4000| 1000|   5000|
+---+------+------+-----+-------+



In [43]:
df.createOrReplaceTempView("emps")

In [45]:
df_result = spark.sql("SELECT * FROM emps")
df_result.show()

+---+------+------+-----+
| id|  name|salary|bonus|
+---+------+------+-----+
|  1|maheer|  2000|  500|
|  2|  mahi|  4000| 1000|
+---+------+------+-----+



In order to use the above function in spark sql, you need to register with spark.udf.register()

In [47]:
def total_pay(s,b):
  return s+b
spark.udf.register(name='Total_Pay_SQL', f = total_pay,returnType=IntegerType() )

<function __main__.total_pay(s, b)>

In [48]:
df_result = spark.sql("SELECT id, name, Total_Pay_SQL(salary, bonus) FROM emps")
df_result.show()

+---+------+----------------------------+
| id|  name|Total_Pay_SQL(salary, bonus)|
+---+------+----------------------------+
|  1|maheer|                        2500|
|  2|  mahi|                        5000|
+---+------+----------------------------+



In [49]:
df_result = spark.sql("SELECT id, name, Total_Pay_SQL(salary, bonus) as TP FROM emps")
df_result.show()

+---+------+----+
| id|  name|  TP|
+---+------+----+
|  1|maheer|2500|
|  2|  mahi|5000|
+---+------+----+

